In [27]:
import os
os.environ.setdefault("USER_AGENT", "AI Agents and Agentic Workflows educational RAG notebook")

from langchain_community.document_loaders import WikipediaLoader, WebBaseLoader, Docx2txtLoader, PyPDFLoader, TextLoader, DirectoryLoader

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama

## Setting up vector database and embeddings

In [28]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
embeddings_model = None  # Use Chroma's local all-MiniLM-L6-v2 embeddings
vector_db = Chroma("tourist_info", embeddings_model)

> **Chroma defaults used here (verified with the installed Chroma 1.3.0):** For a newly created local/single-node collection, `embeddings_model = None` lets Chroma attach its built-in `DefaultEmbeddingFunction`, which uses ONNX Runtime with `all-MiniLM-L6-v2`. The main dense-vector index is **HNSW** (approximate nearest-neighbor search), not IVF or PQ. Its default `space` is **`l2`**, which Chroma defines as squared Euclidean distance: $\sum_i (A_i-B_i)^2$. Therefore, smaller returned distances mean closer matches; this is not cosine similarity or dot product. Newly added vectors first enter a small brute-force buffer (default batch size: 100) before being merged into HNSW. These settings are established when the collection is created.

> Sources: [Chroma index configuration](https://docs.trychroma.com/docs/collections/configure) and [Chroma collection defaults](https://cookbook.chromadb.dev/core/collections/).

In [29]:
try:
    wikipedia_loader = WikipediaLoader(query="Paestum")
    wikipedia_chunks = text_splitter.split_documents(wikipedia_loader.load())
    vector_db.add_documents(wikipedia_chunks)
except Exception as error:
    print(f"Wikipedia API failed ({type(error).__name__}: {error}). Loading the Paestum page directly.")
    wikipedia_loader = WebBaseLoader(
        "https://en.wikipedia.org/wiki/Paestum",
        header_template={"User-Agent": "AI Agents and Agentic Workflows educational RAG notebook"}
    )
    wikipedia_chunks = text_splitter.split_documents(wikipedia_loader.load())
    vector_db.add_documents(wikipedia_chunks)

Wikipedia API failed (JSONDecodeError: Expecting value: line 1 column 1 (char 0)). Loading the Paestum page directly.


In [30]:
word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_chunks = text_splitter.split_documents(word_loader.load())
vector_db.add_documents(word_chunks)

['00c89e8c-d465-4961-8f70-76789143f0b0',
 'cdfee7d2-63d1-43b2-ac84-51886f5176f0',
 '7d7a36f6-e070-4c07-a631-93352a6eba06',
 '91a0c04d-a90e-4229-85a6-d2bd01249918',
 '3b792bc6-1cb5-4a64-98c3-1b7a9c508d67',
 'aa680bff-c0a9-46b8-be6f-204baf020eab',
 'd3c7c65d-7483-4f16-b765-20646b5d7a22',
 'f4c471bd-a67e-46fc-9954-79398aa69faa']

In [31]:
pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_chunks = text_splitter.split_documents(pdf_loader.load())
vector_db.add_documents(pdf_chunks)

['470c11b0-7111-447c-9303-22fda511f472',
 '7459d51c-7440-4bd5-a366-cf8762fb4560',
 '111df7d1-a7cd-47fe-92b8-6a4eeb102ea4',
 'c99a68c9-bce8-45bd-8a1d-7fda9703d97d',
 '46ae8c32-b99b-4980-bf84-28f3bb4ebebb',
 '0f9e4fdb-3fdc-4139-b90e-6178177d17ac',
 '76118353-bf6b-45a8-8190-6ab7cb349fef',
 '30e94fe8-3f9b-4300-8aaa-b113471a0db3',
 'e00e8842-48fc-49bb-ad35-a87039a8fd0e',
 '77de6f63-9e00-4235-bbf5-75e5b5d6aba4',
 '1551a045-4d6b-4134-9b8d-fcb52330f914',
 'c61f93f3-d303-48eb-9e1a-d15049cf4f2d',
 '072b4d5f-fb57-415b-972f-0c4741de39e3',
 'e79827d2-735e-40f0-91fb-1b753cb097de',
 '62b5a10c-a370-4f93-82aa-c24f6c606d61',
 '2d53a28c-34fc-4e9d-8941-37311fd61c60',
 'c81c7bda-1545-4ef0-8a5b-c714b9fcf9c7',
 'd387c04d-b221-434d-a2b1-617348853db1',
 'faf17c8d-ac19-4018-aaf7-5ea7d2cd4edd',
 'b84564c6-42e5-49cb-a657-f9269450ddcd',
 '710de543-f9d1-4681-9651-7574201a7bd1',
 '8c87f4d0-ba1b-4169-9e6b-e192ebebc855',
 'a62790e3-230d-455a-9746-be82be999253',
 'a2dcd13b-acef-4705-ac38-bb0389515ee4',
 '35ea3271-b916-

In [32]:
txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_chunks = text_splitter.split_documents(txt_loader.load())
vector_db.add_documents(txt_chunks)

['4bf4158b-99c7-495b-a130-4b0891b8addc']

## Removing duplication

In [33]:
def split_and_import(loader):
     chunks = text_splitter.split_documents(loader.load())
     vector_db.add_documents(chunks)
     print(f"Ingested chunks created by {loader}")

In [34]:
try:
    wikipedia_loader = WikipediaLoader(query="Paestum")
    split_and_import(wikipedia_loader)
except Exception as error:
    print(f"Wikipedia API failed ({type(error).__name__}: {error}). Loading the Paestum page directly.")
    wikipedia_loader = WebBaseLoader(
        "https://en.wikipedia.org/wiki/Paestum",
        header_template={"User-Agent": "AI Agents and Agentic Workflows educational RAG notebook"}
    )
    split_and_import(wikipedia_loader)

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
split_and_import(word_loader)

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
split_and_import(pdf_loader)

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
split_and_import(txt_loader)

Ingested chunks created by <langchain_community.document_loaders.wikipedia.WikipediaLoader object at 0x000002D93E7529E0>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x000002D93C57A650>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x000002D93C57B750>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x000002D9413849E0>


## Ingesting Multiple Documents from a Folder (two techniques)

### 1) Iterating over all files in a folder

In [35]:
loader_classes = {
    'docx': Docx2txtLoader,
    'pdf': PyPDFLoader,
    'txt': TextLoader
}

In [36]:
import os

def get_loader(filename):
    _, file_extension = os.path.splitext(filename) #A Extract the file extension
    file_extension = file_extension.lstrip('.') #B Remove the leading dot from the extension

    loader_class = loader_classes.get(
        file_extension) #C Get the loader class from the dictionary

    if loader_class:
        return loader_class(filename) #D Instantiate and return the correct loader
    else:
        raise ValueError(f"No loader available for file extension '{file_extension}'")

### Ingesting the files from the folder

In [37]:
folder_path = "CilentoTouristInfo" #A Path to the folder containing the documents

for filename in os.listdir(folder_path): #B iterate over the files in the path
    file_path = os.path.join(folder_path, filename) #C Construct the full path to the file

    if os.path.isfile(file_path): #D Check if it is a file (not a directory)
        try:
            loader = get_loader(file_path) #E Instantiate the correct loader for the file
            print(f"Loader for {filename}: {loader}")
            split_and_import(loader) #F Split and ingest
        except ValueError as e:
            print(e)

Loader for Acciaroli.pdf: <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x000002D93C559310>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x000002D93C559310>
Loader for Cape Palinuro.txt: <langchain_community.document_loaders.text.TextLoader object at 0x000002D9405B4350>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x000002D9405B4350>
Loader for Casalvelino.txt: <langchain_community.document_loaders.text.TextLoader object at 0x000002D940DD0B50>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x000002D940DD0B50>
Loader for Cilentan coast.docx: <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x000002D93C559310>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x000002D93C559310>
Loader for Cilento Coast Map and Travel Guide.docx: <langchain_community.docum

### 2) Ingesting all files with DirectoryLoader

In [12]:
# Unstructured's local PDF inference extra does not support Windows Python 3.13.
# Use Unstructured locally for DOCX/TXT and the existing PyPDFLoader for PDF files.
# Requires: unstructured[docx]
# https://docs.langchain.com/oss/python/integrations/providers/unstructured
# https://docs.unstructured.io/open-source/installation/full-installation
folder_path = "CilentoTouristInfo"

unstructured_directory_loader = DirectoryLoader(
    folder_path, ["**/*.docx", "**/*.txt"]
) #A Load DOCX and TXT files with Unstructured
pdf_directory_loader = DirectoryLoader(
    folder_path, "**/*.pdf", loader_cls=PyPDFLoader
) #B Load PDFs locally with PyPDFLoader

split_and_import(unstructured_directory_loader)
split_and_import(pdf_directory_loader)

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


Ingested chunks created by <langchain_community.document_loaders.directory.DirectoryLoader object at 0x0000020925BB3CB0>
Ingested chunks created by <langchain_community.document_loaders.directory.DirectoryLoader object at 0x00000209257E4410>


> **Windows/Python compatibility note:** Unstructured's local PDF inference dependency is not available for Windows Python 3.13. Therefore, this notebook uses Unstructured locally for DOCX/TXT files and the existing `PyPDFLoader` for PDFs. For full Unstructured PDF/OCR processing, use Python 3.12, install the PDF extra, and provide the required system tools such as Poppler and Tesseract. See the [LangChain Unstructured integration](https://docs.langchain.com/oss/python/integrations/providers/unstructured) and [Unstructured full-installation guide](https://docs.unstructured.io/open-source/installation/full-installation).
>
> Repeated `libmagic is unavailable` messages are non-fatal file-type-detection advisories. This notebook supplies explicit `.docx`, `.txt`, and `.pdf` glob patterns, so files with correct extensions can still be loaded without native `libmagic`. The two `Ingested chunks created by ... DirectoryLoader` messages confirm that both loader paths—Unstructured for DOCX/TXT and `PyPDFLoader` for PDF—completed successfully. Native `libmagic` is mainly useful here for files with missing, incorrect, or ambiguous extensions.

## Querying the vector store directly

In [38]:
query = "Where was Poseidonia and who renamed it to Paestum?"
results = vector_db.similarity_search(query, 4) # four clostest results
print(results)

[Document(id='82bb2cee-e7d0-47bf-bdbf-52b35a165575', metadata={'language': 'en', 'source': 'https://en.wikipedia.org/wiki/Paestum', 'title': 'Paestum - Wikipedia'}, page_content='The Greek settlers who founded the city originally named it Poseidonia (Ancient Greek: Ποσειδωνία). It was eventually conquered by the local Lucanians and later the Romans. The Lucanians renamed it to Paistos and the Romans gave the city its current name.[5]\nAncient ruins and features[edit]\nAerial view of Paestum, looking north; two Hera Temples in foreground, Athena Temple in background.'), Document(id='26fac21f-bc35-4ddc-a280-2c6e2136a7a3', metadata={'source': 'https://en.wikipedia.org/wiki/Paestum', 'language': 'en', 'title': 'Paestum - Wikipedia'}, page_content='The Greek settlers who founded the city originally named it Poseidonia (Ancient Greek: Ποσειδωνία). It was eventually conquered by the local Lucanians and later the Romans. The Lucanians renamed it to Paistos and the Romans gave the city its curr

In [39]:
len(results)

4

## Asking a question through a LangChain's RAG chain

In [40]:
from langchain_core.prompts import PromptTemplate

rag_prompt_template = """Use the following pieces of context
to answer the question at the end.
If you don't know the answer, just say that you don't know,
don't try to make up an answer.
Use three sentences maximum and keep the
answer as concise as possible.
{context}
Question: {question}
Helpful Answer:"""

rag_prompt = PromptTemplate.from_template(rag_prompt_template)

Alternatively, you can pull the prompt instance directly from the **[LangChain Hub](https://smith.langchain.com/hub)**:

```Python
from langchain import hub
rag_prompt = hub.pull("rlm/rag-prompt")
```

In [41]:
retriever = vector_db.as_retriever()

In [42]:
from langchain_core.runnables import RunnablePassthrough
question_feeder = RunnablePassthrough()

`RunnablePassthrough` is a core component in LangChain's LangChain Reference Expression Language (LCEL) that _takes an input and **returns it completely unchanged**_. It acts like an identity function, making it essential for passing raw data—like a user's original question—alongside processed intermediate data in complex multi-step chains.
- **Retrieval-Augmented Generation (RAG) Use Case**: Pass the original user query straight through to a prompt template while a separate retriever branch fetches context.

In [43]:
chatbot = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=65536,
    temperature=0,
    reasoning=False
)

In [44]:
# set up RAG chain

rag_chain = {"context": retriever,
             "question": question_feeder} | rag_prompt | chatbot

Each block in a LangChain chain implements the `Runnable interface` and **accepts a dictionary as input**. This is why the first block in the chain is a dictionary.

In [45]:
def execute_chain(chain, question):
    answer = chain.invoke(question)
    return answer

In [46]:
question = """Where was Poseidonia and who renamed
it to Paestum. Also tell me the source."""
answer = execute_chain(rag_chain, question)
print(answer.content)

Poseidonia was the original name given to the city by Greek settlers, which is now known as Paestum. The Romans gave the city its current name after it was conquered from the Lucanians, who had renamed it Paistos. The source for this information is Wikipedia.


In [47]:
print(answer)

content='Poseidonia was the original name given to the city by Greek settlers, which is now known as Paestum. The Romans gave the city its current name after it was conquered from the Lucanians, who had renamed it Paistos. The source for this information is Wikipedia.' additional_kwargs={} response_metadata={'model': 'gemma4:12b-it-q8_0', 'created_at': '2026-08-09T05:44:13.8153512Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6311314700, 'load_duration': 4653328800, 'prompt_eval_count': 723, 'prompt_eval_duration': 295815000, 'eval_count': 57, 'eval_duration': 1357541000, 'logprobs': None, 'model_name': 'gemma4:12b-it-q8_0', 'model_provider': 'ollama'} id='lc_run--019fe50c-c64f-71f1-b500-b83c8b9df054-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 723, 'output_tokens': 57, 'total_tokens': 780}


In [48]:
# Follow-up question to check if there is any memory of the previous question and answer.

question = """And then, what they do?
Tell me only if you know.
Also tell me the source"""
answer = execute_chain(rag_chain, question)
print(answer.content)

I do not know what "they" do because the provided text does not specify a subject for that action.


It is obvious that the ***chatbot has no memory of previous dialog context*** and doesn’t understand that “they” refers to the Romans. Currently, it’s **stateless** and simply passes questions from the user to the LLM and back, without retaining any memory of the conversation flow. Let’s now add the memory to the chatbot.

## Chatbot memory of message history

In [49]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant, world-class expert in Roman and Greek history, especially in towns located in southern Italy. Provide interesting insights on local history and recommend places to visit with knowledgeable and engaging answers. Answer all questions to the best of your ability, but only use what has been provided in the context. If you don't know, just say you don't know. Use three sentences maximum and keep the answer as concise as possible."),
        ("placeholder", "{chat_history_messages}"),
        ("assistant", "{retrieved_context}"),
        ("human", "{question}"),
    ]
)

retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()
chatbot = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=65536,
    temperature=0,
    reasoning=False
)
chat_history_memory = ChatMessageHistory()

def get_messages(x):
    return chat_history_memory.messages

rag_chain = {
    "retrieved_context": retriever,
    "question": question_feeder,
    "chat_history_messages": RunnableLambda(get_messages)
} | rag_prompt | chatbot

def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'Full chat message history: {chat_history_memory.messages}\n\n')
    return answer

`RunnablePassthrough` takes an input and returns it completely unchanged, whereas `RunnableLambda` wraps a custom function or lambda expression to process, alter, or transform the input data inside a LangChain Expression Language (LCEL) pipeline.

**Core Differences**
- `RunnablePassthrough`: Acts as an identity function (`f(x) = x`). It forwards data untouched, typically used in parallel blocks to carry original inputs forward alongside transformed data or to assign new keys via .assign().
- `RunnableLambda`: Executes custom user logic. It converts any regular Python or JavaScript function into a valid pipeline component to change the data format, parse text, or call external APIs.

**Key Features**

***RunnablePassthrough***
- **Passes raw data**: Outputs whatever input it receives without executing code.
- **Retains context**: Keeps the initial user prompt or variable available for later steps in a chain.
- **Supports assignment**: Includes an `.assign()` method to add new calculated fields into an existing input dictionary.

***RunnableLambda***
- **Transforms data**: Applies custom code, filters, or calculations to the incoming value.
- **Wraps functions**: Turns standard functions into unified components that support `.invoke()`, `.batch()`, and async calls.
- **Handles logic**: Ideal for data validation, formatting strings, or routing decisions inside a sequence.

In [50]:
question = """Where was Poseidonia and who renamed
it to Paestum? Also tell me the source."""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed\nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was founded by Greek settlers in the location now known as Paestum. The Lucanians renamed it to Paistos, and the Romans eventually gave the city its current name. This information is sourced from Wikipedia.', additional_kwargs={}, response_metadata={'model': 'gemma4:12b-it-q8_0', 'created_at': '2026-08-09T05:44:34.4972247Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1996590100, 'load_duration': 212339900, 'prompt_eval_count': 781, 'prompt_eval_duration': 465954000, 'eval_count': 46, 'eval_duration': 1075643000, 'logprobs': None, 'model_name': 'gemma4:12b-it-q8_0', 'model_provider': 'ollama'}, id='lc_run--019fe50d-27f3-70c2-8fb7-ecc451776270-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 781, 'output_tokens': 46, 'total_tokens': 827})]


Poseidon

In [51]:
# Follow-up question to check if there is any memory of the previous question and answer.

question = """And then what did they do?
Also tell me the source"""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed\nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was founded by Greek settlers in the location now known as Paestum. The Lucanians renamed it to Paistos, and the Romans eventually gave the city its current name. This information is sourced from Wikipedia.', additional_kwargs={}, response_metadata={'model': 'gemma4:12b-it-q8_0', 'created_at': '2026-08-09T05:44:34.4972247Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1996590100, 'load_duration': 212339900, 'prompt_eval_count': 781, 'prompt_eval_duration': 465954000, 'eval_count': 46, 'eval_duration': 1075643000, 'logprobs': None, 'model_name': 'gemma4:12b-it-q8_0', 'model_provider': 'ollama'}, id='lc_run--019fe50d-27f3-70c2-8fb7-ecc451776270-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 781, 'output_tokens': 46, 'total_tokens': 827}), HumanMessa

## Verifying conversational retrieval separately from memory

The saved follow-up answer confirms that **message memory works for answer generation**: the model understands that "they" refers to the people discussed in the preceding exchange and that the question concerns events after the renaming. Its refusal is appropriate grounded behavior because the retrieved context does not provide the requested later actions. The [current book notebook](https://github.com/roberto-inf/building-llm-applications/blob/main/ch07/07-QA_across_documents.ipynb) now records the same conservative result with `gpt-5-nano`, so this outcome is not evidence that the local model lacks relevant internal knowledge.

However, this chain is **not history-aware at the retrieval stage**. LangChain retrievers accept a string query and return matching documents; here, `retriever` receives only the literal current question, such as "And then what did they do?", while `chat_history_messages` is supplied separately to the final model prompt. Consequently, the model can resolve the pronoun from memory, but Chroma searches using an ambiguous query that does not mention the Romans, Poseidonia, Paestum, or the renaming. See the [LangChain retriever documentation](https://docs.langchain.com/oss/python/integrations/retrievers).

A production conversational RAG pipeline should first **rewrite the follow-up as a standalone search query** using the relevant message history, retrieve with that explicit query, and only then generate an answer grounded in the retrieved documents. It should also manage per-session history with a dedicated history wrapper such as `RunnableWithMessageHistory`. The current helper is suitable for illustrating the concept, but it adds the current user message before invoking the chain, so that question appears both in the history placeholder and again in the final human-message slot.

The following diagnostics isolate retrieval from generation. They compare exactly what the current chain retrieves for the ambiguous follow-up with what the same retriever returns for an equivalent standalone query. Run both cells after creating and populating `vector_db`.

In [52]:
# Inspect the documents retrieved from the literal, ambiguous follow-up.
follow_up_question = "And then what did they do? Also tell me the source"
literal_follow_up_docs = retriever.invoke(follow_up_question)

def print_retrieved_documents(label, documents):
    print(f"{label}: {len(documents)} document(s)\n")
    for number, document in enumerate(documents, start=1):
        source = document.metadata.get("source", "Unknown source")
        print(f"--- Retrieved document {number} ---")
        print(f"Source: {source}")
        print(document.page_content[:1000])
        print()

print_retrieved_documents(
    "Literal follow-up retrieval", literal_follow_up_docs
)

Literal follow-up retrieval: 4 document(s)

--- Retrieved document 1 ---
Source: CilentoTouristInfo\Parmenides.docx
false and deceitful.

--- Retrieved document 2 ---
Source: CilentoTouristInfo\Parmenides.docx
false and deceitful.

--- Retrieved document 3 ---
Source: https://en.wikipedia.org/wiki/Red-figure_pottery
from oxygen.

--- Retrieved document 4 ---
Source: CilentoTouristInfo\Velia.pdf
•  
Drachma, circa 535-510 BC 
  
•  
Stater struck 334-300 BC 
  
•  
Silver coin from Velia, circa 280 BC, with Athena on the obverse, and a lion devouring a stag on 
the reverse



In [53]:
# Simulate the query that a history-aware rewriting stage should produce.
standalone_follow_up_query = (
    "What did the Romans do after renaming Poseidonia to Paestum? "
    "Also identify the source."
)
standalone_follow_up_docs = retriever.invoke(standalone_follow_up_query)

print(f"Standalone query: {standalone_follow_up_query}\n")
print_retrieved_documents(
    "Standalone-query retrieval", standalone_follow_up_docs
)

Standalone query: What did the Romans do after renaming Poseidonia to Paestum? Also identify the source.

Standalone-query retrieval: 4 document(s)

--- Retrieved document 1 ---
Source: https://en.wikipedia.org/wiki/Paestum
Paestum was established around 600 BCE by settlers from Sybaris, a Greek colony in southern Italy, under the name of Poseidonia (Ancient Greek: Ποσειδωνία). The city thrived as a Greek settlement for about two centuries, witnessing the development of democracy. In 400 BCE, the Lucanians seized the city. Romans took over in 273 BCE, renaming it Paestum and establishing a Latin colony. Later, its decline ensued from shifts in trade routes and the onset of flooding and marsh formation. As Pesto or

--- Retrieved document 2 ---
Source: https://en.wikipedia.org/wiki/Paestum
The Greek settlers who founded the city originally named it Poseidonia (Ancient Greek: Ποσειδωνία). It was eventually conquered by the local Lucanians and later the Romans. The Lucanians renamed it to

### How to interpret the evidence

The executed diagnostics provide decisive evidence about this pipeline:

- **Conversation memory works for answer generation.** In the follow-up response, Gemma resolves the ambiguous pronoun "they" as the Lucanians or Romans from the preceding exchange.
- **Retrieval is not history-aware.** Searching with the literal follow-up retrieved unrelated fragments from `Parmenides.docx`, the Wikipedia article about red-figure pottery, and `Velia.pdf`. The retriever received only the current ambiguous question; it did not receive or rewrite it using the chat history.
- **The indexed corpus contains a relevant partial answer.** Searching with the standalone query retrieved Paestum passages stating that the Romans took over in 273 BCE, renamed the city Paestum, and established a Latin colony.
- **Gemma's refusal was appropriately grounded.** That relevant passage was not among the documents retrieved for the literal follow-up, so answering from internal model knowledge would have violated the instruction to use only the provided context. The result therefore does not demonstrate weaker internal knowledge than the OpenAI model; it demonstrates a retrieval-query limitation.

The recorded run was also not a clean first-ingestion-only test. Its execution history shows that the original Paestum ingestion cells, the alternative Paestum ingestion block, and the `CilentoTouristInfo` ingestion loop were executed. This explains the unrelated Cilento results and repeated identical Paestum chunks. For a clean comparison, restart the kernel to clear the in-memory Chroma collection and message history, run only one ingestion approach, and then rerun the RAG, memory, and diagnostic cells.

**Final conclusion:** the current chain successfully demonstrates generation-side conversational memory, but it is not yet a complete conversational RAG implementation. A production-quality pipeline should use the relevant message history to rewrite each follow-up into a standalone query before retrieval, then generate an answer grounded only in the documents retrieved for that rewritten query.

## Tracing with LangSmith

Stop the notebook and open a new operative system shell (for example Windows command shell).

Configure the relevant environment variables in the OS shell, the rerun the previous Jupyter cells:
```
(env_ch07) C:\...\ch07>set LANGSMITH_TRACING=true
(env_ch07) C:\...\ch07>set LANGSMITH_ENDPOINT=https://api.smith.langchain.com
(env_ch07) C:\...\ch07>set LANGSMITH_PROJECT=Q & A chatbot
(env_ch07) C:\...\ch07>set LANGSMITH_API_KEY=<YOUR_LANGSMITH_API_KEY>
```
Then Restart the Jupyter notebook:

```(env_ch07) C:\...\ch07>jupyter notebook 07-QA_across_documents.ipynb```

Finally re-execute the whole Jupyter notebook cell by cell. All the activity will have not been logged through LangSmith.